In [1]:
import pandas as pd

In [2]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ----- ---------------------------------- 1.8/12.8 MB 11.7 MB/s eta 0:00:01
     --------------- ------------------------ 5.0/12.8 MB 13.7 MB/s eta 0:00:01
     ------------------------- -------------- 8.1/12.8 MB 14.5 MB/s eta 0:00:01
     ----------------------------------- --- 11.5/12.8 MB 14.9 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 14.9 MB/s  0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


# Data Acquisition

In [3]:
path = "../data/Resume.csv"

resume_df = pd.read_csv(path)

resume_df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [4]:
resume_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           2484 non-null   int64 
 1   Resume_str   2484 non-null   object
 2   Resume_html  2484 non-null   object
 3   Category     2484 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB


In [5]:
path = "../data/training_data.csv"

job_df = pd.read_csv(path)

job_df.head()

,company_name,job_description,position_title,description_length,model_response
0,Google,minimum qualifications\nbachelors degree or eq...,Sales Specialist,2727,"{\n ""Core Responsibilities"": ""Responsible fo..."
1,Apple,description\nas an asc you will be highly infl...,Apple Solutions Consultant,828,"{\n ""Core Responsibilities"": ""as an asc you ..."
2,Netflix,its an amazing time to be joining netflix as w...,Licensing Coordinator - Consumer Products,3205,"{\n ""Core Responsibilities"": ""Help drive bus..."
3,Robert Half,description\n\nweb designers looking to expand...,Web Designer,2489,"{\n ""Core Responsibilities"": ""Designing webs..."
4,TrackFive,at trackfive weve got big goals were on a miss...,Web Developer,3167,"{\n ""Core Responsibilities"": ""Build and layo..."


In [6]:
job_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 853 entries, 0 to 852
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   company_name        853 non-null    object
 1   job_description     853 non-null    object
 2   position_title      853 non-null    object
 3   description_length  853 non-null    int64 
 4   model_response      853 non-null    object
dtypes: int64(1), object(4)
memory usage: 33.4+ KB


In [7]:
job_df['model_response'][1]

' {\n  "Core Responsibilities": "as an asc you will be highly influential in growing mind and market share of apple products while building longterm relationships with those who share your passion customer experiences are driven through you and your partner team growing in an ever changing and challenging environment you strive for perfection whether its maintaining visual merchandising or helping to grow and develop your partner team",\n  "Required Skills": "a passion to help people understand how apple products can enrich their livesexcellent communication skills allowing you to be as comfortable in front of a small group as you are speaking with individuals years preferred working in a dynamic sales andor results driven environment as well as proven success developing customer loyaltyability to encourage a partner team and grow apple business",\n  "Educational Requirements": "N/A",\n  "Experience Level": "years preferred",\n  "Preferred Qualifications": "N/A",\n  "Compensation and B

# Data Cleaning

### Missing Values

In [8]:
cols_with_question = resume_df.columns[resume_df.isin(['?']).any()].tolist()
print(cols_with_question)

[]


In [9]:
cols_with_question = job_df.columns[job_df.isin(['?']).any()].tolist()
print(cols_with_question)

[]


In [10]:
print( (job_df.isna().sum() / len(job_df) ) * 100)

company_name          0.0
job_description       0.0
position_title        0.0
description_length    0.0
model_response        0.0
dtype: float64


In [11]:
print( (resume_df.isna().sum() / len(resume_df) ) * 100)

ID             0.0
Resume_str     0.0
Resume_html    0.0
Category       0.0
dtype: float64


### Duplicates

In [12]:
job_df.duplicated().sum()

np.int64(0)

In [13]:
resume_df.duplicated().sum()

np.int64(0)

## Data Preprocessing

### Text Cleaning
- remove html
- remove whitespace
- remove extra chars
- lowercase

In [14]:
from bs4 import BeautifulSoup
import re

def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-z0-9+#.\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

In [15]:
job_df["clean_description"] = job_df["job_description"].apply(clean_text)
resume_df["clean_description"] = resume_df["Resume_str"].apply(clean_text)

In [16]:
print(job_df["clean_description"], '\n')
print(resume_df["clean_description"])

0      minimum qualifications bachelors degree or equ...
1      description as an asc you will be highly influ...
2      its an amazing time to be joining netflix as w...
3      description web designers looking to expand yo...
4      at trackfive weve got big goals were on a miss...
                             ...                        
848    job description parttime make big money at men...
849    responsibilities parkers internship program wa...
850    the borgen project is an innovative national c...
851    put the world on vacation at wyndham destinati...
852    this job handles customer inquiries by telepho...
Name: clean_description, Length: 853, dtype: object 

0       hr administrator marketing associate hr admini...
1       hr specialist us hr operations summary versati...
2       hr director summary over 20 years experience i...
3       hr specialist summary dedicated driven and dyn...
4       hr manager skill highlights hr skills hr depar...
                             

### Keyword Extraction
- skills
- responsibilites
- qualifications
- requirements
- degree
- experience

#### Resume

In [17]:
def extract_degrees(text):
    if not isinstance(text, str):
        return []

    matches = re.findall(
        r"\b(?:high school diploma|associate(?:'s)?|bachelor(?:'s)?|master(?:'s)?|doctorate|ph\.?d\.?|mba)\b",
        text,
        flags=re.IGNORECASE
    )

    # dict for degrees
    degree_map = {
        "high school diploma": "High School Diploma",
        "associate": "Associate",
        "associate's": "Associate",
        "bachelor": "Bachelor",
        "bachelor's": "Bachelor",
        "master": "Master",
        "master's": "Master",
        "doctorate": "Doctorate",
        "phd": "Doctorate",
        "ph.d": "Doctorate",
        "ph.d.": "Doctorate",
        "mba": "MBA"
    }

    standardized = [
        degree_map.get(match.lower(), match.title())
        for match in matches
    ]

    return list(dict.fromkeys(standardized))

In [18]:
resume_df["degrees"] = resume_df["clean_description"].apply(extract_degrees)

In [19]:
resume_df["degrees"][0]

['Associate', 'High School Diploma']

#### Job

In [20]:
job_df['model_response'][0]

' {\n  "Core Responsibilities": "Responsible for expanding Google Workspace product adoption across an assigned territory. Build relationships with customers to understand needs and provide Google Workspace solutions. Partner with account teams to construct solutions and grow business for Google Workspace.",\n  "Required Skills": "Bachelor\'s degree or equivalent experience. Experience managing enterprise SaaS accounts and sales cycles.", \n  "Educational Requirements": "Bachelor\'s degree or equivalent experience.",\n  "Experience Level": "Experience managing enterprise SaaS accounts and sales cycles.",\n  "Preferred Qualifications": "Experience building strategic partnerships with enterprise customers. Ability to work through a reseller ecosystem. Excellent communication and strategic thinking skills.",\n  "Compensation and Benefits": "N/A"\n}'

##### Skill Extraction
- using SPACY
- multiple Nouns in a row?

In [21]:
import spacy

nlp = spacy.load("en_core_web_sm")

stop_phrases = {
    "time",
    "customer",
    "people",
    "part",
    "employment",
    "responsibility",
    "service",
    "client",
    "individual",
    "regard",
    "world",
    "business",
    "development",
    "knowledge",
    "skill",
    "support",
    "disability",
    "member",
    "environment",
    "process",
    "information",
    "office",
    "system",
    "task",
    "project",
    "communication",
}

remove_words = {"a", "an", "the", "their", "this"}

generic_roots = {
    "people",
    "time",
    "part",
    "individual",
    "knowledge",
    "skill",
    "support",
    "business",
    "development",
    "service",
    "customer",
    "world",
}

In [22]:
def extract_skill_candidates(text):
    if not isinstance(text, str) or not text.strip():
        return []

    doc = nlp(text)
    candidates = []

    for chunk in doc.noun_chunks:
        phrase = chunk.text.lower().strip()

        # spaces
        phrase = re.sub(r"\s+", " ", phrase)
        
        # phrases
        word_count = len(phrase.split())
        if not 1 <= word_count <= 5:
            continue

        # roots
        if chunk.root.lemma_.lower() in generic_roots:
            continue
            
        words = phrase.split()
        
        # remove_words
        while words and words[0] in remove_words:
            words.pop(0)
        
        phrase = " ".join(words)

         # lemma
        phrase = " ".join(
            token.lemma_
            for token in nlp(phrase)
            if not token.is_space
        )
        
        if not phrase:
            continue
            
        # pronouns
        if chunk.root.pos_ == "PRON":
            continue
            
        # stop
        if phrase in stop_phrases:
            continue

        # numbers
        if phrase.isnumeric():
            continue

        # letter check
        if not re.search(r"[a-z]", phrase):
            continue

        candidates.append(phrase)

    # duplicates
    return list(dict.fromkeys(candidates))

In [23]:
job_df["skill_candidates"] = job_df["clean_description"].apply(extract_skill_candidates)
job_df[["position_title", "clean_description", "skill_candidates"]].head()

,position_title,clean_description,skill_candidates
0,Sales Specialist,minimum qualifications bachelors degree or equ...,"[minimum qualification bachelor, equivalent pr..."
1,Apple Solutions Consultant,description as an asc you will be highly influ...,"[description, asc, grow mind, market share, ap..."
2,Licensing Coordinator - Consumer Products,its an amazing time to be joining netflix as w...,"[netflix, entertainment, over million pay memb..."
3,Web Designer,description web designers looking to expand yo...,"[description web designer, your professional r..."
4,Web Developer,at trackfive weve got big goals were on a miss...,"[big goal, mission, easytouse tool, platform, ..."


In [24]:
job_df["skill_candidates"][0]

['minimum qualification bachelor',
 'equivalent practical experience year',
 'experience',
 'sale cycle prefer qualification year',
 'strategic business partnership',
 'enterprise customersability',
 'reseller ecosystem',
 'businessability',
 'pitch',
 'territory business strategyability',
 'relationship',
 'result',
 'crossfunctionalmatrixe environmentability',
 'crosspromote and uppromote opportunity',
 'job',
 'google cloud team',
 'lead company school',
 'government agency',
 'google tool',
 'google workspace search',
 'innovative power',
 'our product',
 'organization',
 'your guide light',
 'good solution',
 'innovation',
 'your passion',
 'google product',
 'magic',
 'google',
 'google workspace team',
 'use',
 'application',
 'entrepreneurial team',
 'future',
 'technology',
 'customer employee',
 'partner',
 'google workspace sale specialist',
 'maintenance',
 'expansion',
 'google workspace business growth',
 'region',
 'role',
 'strategy',
 'unique insight',
 'google workspa

##### JSON Structure

In [25]:
import json

text = job_df['model_response'][0]
data = json.loads(text)

In [26]:
parsed = job_df["model_response"].apply(json.loads)
parsed_df = pd.json_normalize(parsed)

job_df = job_df.join(parsed_df)

In [27]:
job_df["degrees"] = job_df["Educational Requirements"].apply(extract_degrees)

In [28]:
job_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 853 entries, 0 to 852
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   company_name               853 non-null    object
 1   job_description            853 non-null    object
 2   position_title             853 non-null    object
 3   description_length         853 non-null    int64 
 4   model_response             853 non-null    object
 5   clean_description          853 non-null    object
 6   skill_candidates           853 non-null    object
 7   Core Responsibilities      853 non-null    object
 8   Required Skills            853 non-null    object
 9   Educational Requirements   853 non-null    object
 10  Experience Level           853 non-null    object
 11  Preferred Qualifications   853 non-null    object
 12  Compensation and Benefits  853 non-null    object
 13  medical specialty          1 non-null      object
 14  schedule  

# Saving

In [29]:
resume_df.to_csv("../data/resume_clean.csv", index=False)
job_df.to_csv("../data/job_clean.csv", index=False)